In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Get initial data
df = pd.DataFrame.from_records(db.read({}))

# Remove _id column to prevent errors
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Print column names and their indices to verify the correct indices for the map
print("Column names and indices:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'  # Make sure this matches your actual file name
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Header with logo and creator info
    html.Div([
        # Add the logo with link to SNHU
        html.A(
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height': '100px', 'width': 'auto'}),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        # Add my name as the creator
        html.Div(html.H3("Created by Jenna Robbins"), style={'textAlign': 'right'})
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center'}),

    html.Center(html.B(html.H1('Grazioso Salvare - Rescue Dog Finder'))),
    html.Hr(),

    # Filter options
    html.Div([
        html.H4("Filter by Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='all',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ], style={'margin': '20px 0'}),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_current=0,
        page_action='native',
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        style_table={'overflowX': 'auto'},
        style_cell={
            'height': 'auto',
            'minWidth': '100px',
            'width': '100px',
            'maxWidth': '300px',
            'whiteSpace': 'normal',
            'textAlign': 'left'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        },
        row_selectable='single'
    ),

    html.Br(),
    html.Hr(),

    # Charts section side by side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            # Pie chart
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            # Map
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter data based on selected rescue type
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Define query based on filter type
    if filter_type == 'water':
        query = {
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'mountain':
        query = {
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'disaster':
        query = {
            'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:  # 'all' or default
        query = {}

    # Get data from database using CRUD module
    results = db.read(query)
    df_filtered = pd.DataFrame.from_records(results)

    # Remove the _id column to prevent errors
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)

    # Return the filtered data
    return df_filtered.to_dict('records')

# Update pie chart based on filtered data
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if not viewData:
        return [html.Div("No data available")]

    dff = pd.DataFrame.from_dict(viewData)

    # Create a pie chart showing breed distribution
    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Breed Distribution',
                hole=0.3,  # Makes it a donut chart for better visualization
                color_discrete_sequence=px.colors.qualitative.Pastel
            )
        )
    ]

# Highlight selected columns in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# Update map based on selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        return
    elif index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    # Only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    try:
        # Try to use the original column indices
        # Column 13 and 14 define the grid-coordinates for the map
        # Column 4 defines the breed for the animal
        # Column 9 defines the name of the animal

        # Check if have enough columns
        if len(dff.columns) > 14:
            lat = dff.iloc[row, 13]
            lon = dff.iloc[row, 14]
            breed = dff.iloc[row, 4]
            name = dff.iloc[row, 9]
        else:
            # If thers not have enough columns, try to find them by name
            lat_col = next((col for col in dff.columns if 'lat' in col.lower()), None)
            lon_col = next((col for col in dff.columns if 'lon' in col.lower() or 'lng' in col.lower()), None)
            breed_col = next((col for col in dff.columns if 'breed' in col.lower()), None)
            name_col = next((col for col in dff.columns if 'name' in col.lower()), None)

            if all([lat_col, lon_col, breed_col, name_col]):
                lat = dff.iloc[row][lat_col]
                lon = dff.iloc[row][lon_col]
                breed = dff.iloc[row][breed_col]
                name = dff.iloc[row][name_col]
            else:
                # If can't find the columns, use default values
                lat = 30.75
                lon = -97.48
                breed = "Unknown"
                name = "Unknown"

        # Convert to float if needed
        lat = float(lat)
        lon = float(lon)

        # Austin TX is at [30.75,-97.48]
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id"),
                # Marker with tool tip and popup
                dl.Marker(position=[lat, lon], children=[
                    dl.Tooltip(breed),
                    dl.Popup([
                        html.H1("Animal Name"),
                        html.P(name)
                    ])
                ])
            ])
        ]
    except Exception as e:
        print(f"Error in map update: {e}")
        # Return a default map if there's an error
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]

# Run the app
app.run_server(debug=True)

TypeError: AnimalShelter.__init__() takes 1 positional argument but 3 were given